### A small notebook to test functions from the simulation directory

In [ ]:
import numpy as np
import pandas as pd
from collections import Counter
from states import sample_next_state, validate_transition_matrix, transition_probabilities
from scipy.special import softmaxañadido
from equations import generate_economic_context, generate_network_exposure, generate_structuring_score, generate_transaction_volume, generate_cross_border_ratio, simulate_account

In [2]:
BASE_TRANSITION = np.array(
    [
        [0.94, 0.05, 0.01],
        [0.10, 0.75, 0.15],
        [0.03, 0.17, 0.80],
    ],
    dtype=float,
)


In [3]:
validate_transition_matrix(BASE_TRANSITION)

In [4]:
bad_matrix = np.array(
    [
        [0.8, 0.1, 0.1],
        [0.5, 0.5, 0.5],
        [0.1, 0.2, 0.7],
    ]
)

In [5]:
validate_transition_matrix(bad_matrix)

ValueError: Each row of the transition matrix must sum to 1.

In [6]:
###cada nuevo estado modifica las probabilidades del siguiente.
###Eso crea persistencia temporal


rng = np.random.default_rng(42)

state = 0

for _ in range(20):
    state = sample_next_state(
        previous_state=state,
        transition_matrix=BASE_TRANSITION,
        rng=rng,
    )

    print(state)

0
0
0
0
0
1
1
1
1
1
1
2
2
2
2
2
2
1
1
1


In [7]:
rng = np.random.default_rng(42)

state = 0
trajectory = [state]

for _ in range(100):
    state = sample_next_state(
        previous_state=state,
        transition_matrix=BASE_TRANSITION,
        rng=rng,
    )

    trajectory.append(state)

print(trajectory)

[0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 2, 2, 2, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]


In [8]:
unique, counts = np.unique(
    trajectory,
    return_counts=True,
)

print(dict(zip(unique, counts)))

{np.int64(0): np.int64(58), np.int64(1): np.int64(34), np.int64(2): np.int64(9)}


In [9]:
probabilities = np.array([0.94, 0.05, 0.01])

logits = np.log(probabilities)

recovered = softmax(logits)

print(logits)
print(recovered)

[-0.0618754  -2.99573227 -4.60517019]
[0.94 0.05 0.01]


In [10]:
for exposure in [0.0, 0.25, 0.50, 0.75, 1.0]:
    probabilities = transition_probabilities(
        previous_state=0,
        network_exposure=exposure,
    )

    print(
        f"Exposure={exposure:.2f}: "
        f"{np.round(probabilities, 4)}"
    )

TypeError: transition_probabilities() missing 1 required positional argument: 'network_exposure_effect'

In [11]:
NETWORK_EXPOSURE_EFFECT = np.array(
    [-1.0, 0.25, 1.25],
    dtype=float,
)

In [12]:
for exposure in [0.0, 0.25, 0.50, 0.75, 1.0]:
    probabilities = transition_probabilities(
        previous_state=0,
        network_exposure=exposure, network_exposure_effect=NETWORK_EXPOSURE_EFFECT
    )

    print(
        f"Exposure={exposure:.2f}: "
        f"{np.round(probabilities, 4)}"
    )

Exposure=0.00: [0.94 0.05 0.01]
Exposure=0.25: [0.9163 0.0666 0.0171]
Exposure=0.50: [0.8833 0.0878 0.0289]
Exposure=0.75: [0.838  0.1138 0.0482]
Exposure=1.00: [0.7772 0.1443 0.0785]


In [14]:
for previous_state in [0, 1, 2]:
    print(f"\nPrevious state = {previous_state}")

    for exposure in [0.0, 0.5, 1.0]:
        probabilities = transition_probabilities(
            previous_state=previous_state,
            network_exposure=exposure,
            network_exposure_effect=NETWORK_EXPOSURE_EFFECT
        )

        print(
            f"Exposure={exposure:.1f}: "
            f"{np.round(probabilities, 4)}"
        )


Previous state = 0
Exposure=0.0: [0.94 0.05 0.01]
Exposure=0.5: [0.8833 0.0878 0.0289]
Exposure=1.0: [0.7772 0.1443 0.0785]

Previous state = 1
Exposure=0.0: [0.1  0.75 0.15]
Exposure=0.5: [0.0509 0.7137 0.2353]
Exposure=1.0: [0.0241 0.6322 0.3437]

Previous state = 2
Exposure=0.0: [0.03 0.17 0.8 ]
Exposure=0.5: [0.0107 0.113  0.8764]
Exposure=1.0: [0.0037 0.0722 0.9241]


In [16]:
def test_generate_economic_context_returns_float():
    rng = np.random.default_rng(42)

    context = generate_economic_context(
        previous_context=0.0,
        rng=rng,
    )

    assert isinstance(context, float)

In [17]:
test_generate_economic_context_returns_float()

In [18]:
rng = np.random.default_rng(42)

economic_context = 0.0

for t in range(10):
    economic_context = generate_economic_context(
        previous_context=economic_context,
        rng=rng,
    )

    print(t, economic_context)

0 0.15235853987721568
1 -0.39810522121847525
2 0.056741420928448416
3 0.5156754949383656
4 -0.5629771983762257
5 -1.1014715121321397
6 -0.8172570081220691
7 -0.8119269026694464
8 -0.6579421008877016
9 -0.9528756444969513


In [20]:
generate_network_exposure(
    economic_context=economic_context,
    rng=rng,
)

0.10463794361831995

In [21]:
rng = np.random.default_rng(42)

economic_context = 0.0

for t in range(10):
    economic_context = generate_economic_context(
        previous_context=economic_context,
        rng=rng,
    )

    network_exposure = generate_network_exposure(
        economic_context=economic_context,
        rng=rng,
    )

    print(
        t,
        round(economic_context, 3),
        round(network_exposure, 3),
    )

0 0.152 0.069
1 0.497 0.28
2 -0.578 0.033
3 -0.398 0.073
4 -0.327 0.054
5 0.178 0.212
6 0.175 0.255
7 0.374 0.091
8 0.484 0.092
9 0.826 0.202


In [22]:
import numpy as np

from aml_working_paper.simulation.equations import (
    generate_economic_context,
    generate_network_exposure,
)

from aml_working_paper.simulation.states import sample_next_state


rng = np.random.default_rng(42)

economic_context = 0.0
previous_aml_state = 0

for t in range(20):
    economic_context = generate_economic_context(
        previous_context=economic_context,
        rng=rng,
    )

    network_exposure = generate_network_exposure(
        economic_context=economic_context,
        rng=rng,
    )

    aml_state = sample_next_state(
        previous_state=previous_aml_state,
        network_exposure=network_exposure,
        rng=rng,
    )

    print(
        t,
        round(economic_context, 3),
        round(network_exposure, 3),
        aml_state,
    )

    previous_aml_state = aml_state

0 0.152 0.069 0
1 0.592 0.053 1
2 0.538 0.143 1
3 0.004 0.201 2
4 0.036 0.235 2
5 -0.401 0.113 1
6 0.118 0.126 1
7 -0.246 0.207 2
8 -0.411 0.071 2
9 -0.146 0.139 2
10 0.954 0.179 2
11 0.356 0.217 2
12 0.228 0.083 2
13 0.508 0.255 2
14 -0.179 0.146 2
15 -0.027 0.126 2
16 0.414 0.181 2
17 0.365 0.182 2
18 -0.437 0.071 2
19 -0.669 0.061 1


In [3]:
import numpy as np

seed_sequence = np.random.SeedSequence(42)

child_seeds = seed_sequence.spawn(6)

rng_economic = np.random.default_rng(child_seeds[0])
rng_network = np.random.default_rng(child_seeds[1])
rng_state = np.random.default_rng(child_seeds[2])
rng_structuring = np.random.default_rng(child_seeds[3])
rng_cross_border = np.random.default_rng(child_seeds[4])
rng_volume = np.random.default_rng(child_seeds[5])

In [ ]:
rng = np.random.default_rng(42)

economic_context = 0.0
previous_aml_state = 0
states = []
rows = []

for t in range(100):
    economic_context = generate_economic_context(
    previous_context=economic_context,
    rng=rng_economic,
    )

    network_exposure = generate_network_exposure(
        economic_context=economic_context,
        rng=rng_network,
    )

    aml_state = sample_next_state(
        previous_state=previous_aml_state,
        network_exposure=network_exposure,
        rng=rng_state,
    )

    structuring_score = generate_structuring_score(
        aml_state=aml_state,
        rng=rng_structuring,
    )

    cross_border_ratio = generate_cross_border_ratio(
        network_exposure=network_exposure,
        aml_state=aml_state,
        rng=rng_cross_border,
    )

    transaction_volume = generate_transaction_volume(
        economic_context=economic_context,
        aml_state=aml_state,
        structuring_score=structuring_score,
        rng=rng_volume,
)

    print(
        t,
        round(network_exposure, 3),
        aml_state,

    )
    rows.append(
    {
        "time": t,
        "economic_context": economic_context,
        "network_exposure": network_exposure,
        "aml_state": aml_state,
        "structuring_score": structuring_score,
        "cross_border_ratio": cross_border_ratio,
        "transaction_volume": transaction_volume,
    }
)
    

    # dentro del loop
    states.append(aml_state)


    previous_aml_state = aml_state
print(Counter(states))

0 0.3 0
1 0.255 0
2 0.091 0
3 0.136 0
4 0.296 0
5 0.123 0
6 0.098 0
7 0.15 0
8 0.178 0
9 0.12 0
10 0.227 0
11 0.164 0
12 0.249 0
13 0.391 0
14 0.216 0
15 0.115 0
16 0.086 0
17 0.324 0
18 0.342 2
19 0.045 2
20 0.03 2
21 0.11 2
22 0.211 2
23 0.066 2
24 0.212 2
25 0.079 2
26 0.049 1
27 0.052 2
28 0.204 2
29 0.072 2
30 0.146 2
31 0.288 2
32 0.135 2
33 0.13 2
34 0.093 1
35 0.135 0
36 0.144 0
37 0.207 0
38 0.144 0
39 0.26 0
40 0.095 0
41 0.229 1
42 0.302 0
43 0.259 0
44 0.219 0
45 0.21 2
46 0.077 2
47 0.081 2
48 0.095 2
49 0.066 2
50 0.045 2
51 0.054 2
52 0.009 2
53 0.097 2
54 0.018 2
55 0.031 1
56 0.139 2
57 0.021 1
58 0.108 1
59 0.07 1
60 0.032 1
61 0.028 1
62 0.052 0
63 0.073 0
64 0.044 0
65 0.02 0
66 0.022 0
67 0.044 0
68 0.034 0
69 0.014 0
70 0.037 0
71 0.108 0
72 0.036 1
73 0.023 0
74 0.165 0
75 0.028 0
76 0.068 0
77 0.086 0
78 0.201 0
79 0.048 0
80 0.183 0
81 0.393 1
82 0.132 1
83 0.205 1
84 0.151 0
85 0.018 0
86 0.096 0
87 0.027 0
88 0.169 0
89 0.073 0
90 0.183 0
91 0.163 0
92 0.15 0

In [22]:
aml_df = pd.DataFrame(rows)

In [23]:
print(aml_df.loc[12:19])

    time  economic_context  network_exposure  aml_state  structuring_score  \
12    12          0.808944          0.248662          0           0.108337   
13    13          0.405639          0.391175          0           0.174038   
14    14         -0.225164          0.215924          0           0.080639   
15    15          0.029497          0.114659          0           0.197006   
16    16          0.556055          0.085981          0           0.208982   
17    17          0.237687          0.324297          0           0.137347   
18    18         -0.597332          0.342471          2           0.608486   
19    19         -1.234778          0.044621          2           0.420965   

    cross_border_ratio  transaction_volume  
12            0.087887           24.041354  
13            0.166761           14.752347  
14            0.038897           17.882747  
15            0.124182           22.995319  
16            0.442060            9.965471  
17            0.069295     

In [24]:
print(aml_df.loc[48:55])

    time  economic_context  network_exposure  aml_state  structuring_score  \
48    48         -0.118494          0.094662          2           0.531012   
49    49         -0.634008          0.065782          2           0.518601   
50    50         -1.379461          0.044884          2           0.531232   
51    51         -1.489580          0.054477          2           0.386358   
52    52         -1.827912          0.008938          2           0.359359   
53    53         -1.038873          0.096720          2           0.703555   
54    54         -1.693543          0.017976          2           0.566235   
55    55         -0.855340          0.031222          1           0.273113   

    cross_border_ratio  transaction_volume  
48            0.205219           17.294054  
49            0.412301           32.975758  
50            0.232984           25.512745  
51            0.296156           19.871152  
52            0.152130           15.163849  
53            0.349667     

In [4]:
df = simulate_account(
    periods=10_000,
    seed=42,
)

In [5]:
correlation = df[
    ["economic_context", "network_exposure"]
].corr()

print(correlation)

                  economic_context  network_exposure
economic_context          1.000000          0.647348
network_exposure          0.647348          1.000000


In [6]:
df["economic_context_bin"] = pd.qcut(
    df["economic_context"],
    q=5,
    labels=False,
)

exposure_by_context = (
    df.groupby("economic_context_bin")[
        "network_exposure"
    ]
    .mean()
)

print(exposure_by_context)

economic_context_bin
0    0.064483
1    0.101667
2    0.137231
3    0.185524
4    0.278581
Name: network_exposure, dtype: float64
